# Training Model — Klasifikasi Ukuran Tanah Longsor (KNN vs Random Forest)

Notebook ini adalah **satu-satunya tempat model produksi dihasilkan**. Backend tidak
melatih model saat runtime; ia hanya memuat artefak hasil notebook ini lalu memakainya
untuk inference. Konsekuensinya, notebook ini harus bisa dibaca sebagai dokumen kerja,
bukan sekadar kumpulan sel yang kebetulan jalan.

Karena itu setiap tahap ditulis dengan pola yang sama:

1. **Apa** yang dilakukan — kodenya.
2. **Kenapa** dipilih begitu — markdown sebelum kode.
3. **Apa arti angkanya** — output yang dicetak, termasuk interpretasinya.

## Artefak yang dihasilkan

| Berkas | Isi | Dibaca oleh |
| --- | --- | --- |
| `artifacts/v1/preprocessor.json` | parameter scaler + kosakata one-hot | backend (inference) |
| `artifacts/v1/knn_store.npz` atau `rf_trees.npz` | model pemenang | backend (inference) |
| `artifacts/v1/rf_trees.npz` / `knn_store.npz` | model pembanding (runner-up) | laporan / analisis |
| `artifacts/v1/model_meta.json` | kontrak: tipe model, hyperparameter, `d_enc`, label kelas, metrik | backend (memilih berkas model) |

`model_meta.json` adalah satu-satunya sumber kebenaran backend untuk tahu berkas model
mana yang dimuat (`artifact_files.model`). Mengganti nama berkas tanpa memperbarui meta
akan membuat backend gagal start — itu disengaja, supaya ketidakcocokan terlihat.

## Prasyarat

- Dijalankan dari `backend/` — notebook mencari `data/*.csv` dan menulis ke `artifacts/`.
- Tepat satu berkas CSV di `backend/data/`.
- Seluruh algoritma memakai implementasi manual NumPy di `app/core`. scikit-learn hanya
  muncul di sel verifikasi sebagai pembanding independen, **bukan** untuk melatih model
  yang diekspor.
- Runtime: ±10 menit bila cache `experiments/*.pkl` masih valid — §6.1 tetap melatih
  ulang 300 pohon; ±3 jam bila grid search dihitung ulang dari nol.


## Peta alur

| Bagian | Isi | Objek kunci |
| --- | --- | --- |
| §0 | Import, konfigurasi, path, definisi grid | `KNN_GRID`, `RF_GRID`, `TEST_SIZES` |
| §1 | Data cleaning (reduksi baris) + diagnostik | `df_clean` |
| §2 | Feature engineering (9 fitur) + `d_enc` | `df_model` |
| §3 | Desain evaluasi anti-leakage | `stratified_holdout`, `stratified_kfold` |
| §4 | Grid search + 5-fold CV per rasio | `knn_reports`, `rf_reports` |
| §5 | Perbandingan, pemilihan, pola kesalahan | `winner`, `primary` |
| §6 | Ekspor artefak sesuai kontrak SPEC | `model_meta.json` |
| §7 | Verifikasi artefak | round-trip + parity |
| §8 | Laporan & kesimpulan | tabel + interpretasi |
| §A | Lampiran: profil biaya komputasi | opsional, tanpa artefak |

Urutannya tidak boleh ditukar: §6 membutuhkan `primary` dari §5, dan §7 membutuhkan
artefak yang baru ditulis §6.

## §0. Konfigurasi dan reproduksibilitas

Semua keputusan eksperimen dikumpulkan di satu tempat supaya mudah diaudit.

- **`SEED = 42`** dipakai untuk split, K-Fold, dan inisialisasi model. Dengan seed tetap
  dan urutan data tetap, hasil bisa direproduksi persis — inilah yang membuat sel
  verifikasi boleh memakai `assert` numerik.
- **`TEST_SIZES = (0.10, 0.15, 0.20)`** — tiga rasio holdout, bukan satu. Tujuannya
  menguji apakah pemilihan model *stabil* terhadap ukuran data uji. Kalau pemenang
  berganti antar rasio, kesimpulannya rapuh dan itu harus terlihat, bukan disembunyikan.
- **`PRIMARY_TEST_SIZE = 0.20`** — rasio yang artefaknya diekspor. Dipilih yang paling
  besar karena memberi estimasi metrik paling stabil (±2.000 baris uji).
- **`N_SPLITS = 5`** — kompromi standar antara biaya dan varians estimasi. Dengan ±8 ribu
  baris latih, tiap fold validasi berisi ±1.600 baris.
- **`N_JOBS = 4`** — Random Forest manual memparalelkan per pohon. Nilai ini hanya
  memengaruhi waktu, bukan hasil: seed tiap pohon tetap, jadi `n_jobs=1` dan `n_jobs=4`
  memberi prediksi identik. Turunkan ke 1 kalau mesin sedang dipakai hal lain.
- **`FAST_MODE`** mengecilkan data dan grid untuk uji cepat alur. **Jangan** dipakai untuk
  produksi: nilainya masuk ke nama folder artefak (`fast-test`), sehingga artefak
  eksperimen tidak akan menimpa `v1`.
- **`FORCE_RERUN = True`** mengabaikan cache `experiments/*.pkl`. Pakai hanya kalau grid
  atau data memang berubah.

Grid search menyimpan hasil per (algoritma, rasio) ke `experiments/`. Cache divalidasi
dengan *signature*: grid, seed, `n_splits`, **jumlah baris data**, dan `fast_mode`. Kalau
salah satu berubah, hasil lama otomatis dianggap kedaluwarsa dan dihitung ulang. Ini
mencegah skenario paling berbahaya — memakai hasil eksperimen lama untuk konfigurasi baru
lalu mengira angka itu valid.


### §0.1 Import

Hanya `app/core` yang diimpor untuk urusan model. Kalau ada fungsi ML yang dibutuhkan tapi
belum ada di sana, tambahkan ke `app/core` — jangan menduplikasinya di notebook, karena
backend memakai kode yang sama.

In [1]:
from pathlib import Path
import sys, json, time, pickle
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import math
from tqdm.auto import tqdm

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from app.core import (
    CATEGORICAL, NUMERIC, TARGET, FEATURE_COLUMNS, CLASS_LABELS,
    clean_raw, engineer_features, payload_to_frame, Preprocessor,
    KNNClassifier, RandomForest, evaluate, encode_target,
    stratified_holdout, stratified_kfold, holdout_grid_search,
)
print("app.core dimuat dari:", Path(sys.modules["app.core"].__file__).parent)

app.core dimuat dari: c:\Kuliah\Semester 7\Penambangan Data dan Analisis\Tugas-Akhir-PDA\backend\app\core


### §0.2 Konfigurasi eksperimen

Nilai di bawah adalah satu-satunya tempat angka eksperimen didefinisikan. Mengubahnya
berarti mengubah signature cache, jadi hasil lama otomatis tidak dipakai.

In [2]:
FAST_MODE = False          # True = data & grid diperkecil untuk uji cepat alur
FORCE_RERUN = False        # True = abaikan cache experiments/*.pkl
N_JOBS = 4                 # paralelisme RF; 1 = deterministik, ramah CPU
SEED = 42                  # seed tunggal untuk split, K-Fold, dan model
N_SPLITS = 5               # jumlah fold CV
TEST_SIZES = (0.10, 0.15, 0.20)   # tiga rasio holdout untuk uji kestabilan
PRIMARY_TEST_SIZE = 0.20   # rasio yang artefaknya diekspor
MODEL_VERSION = "fast-test" if FAST_MODE else "v1"

KNN_GRID = {
    "k": [3, 5, 7, 9, 11, 15, 21, 31, 41],
    "weights": ["uniform", "distance"],
    "metric": ["euclidean", "manhattan", "chebyshev"],
}
RF_GRID = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 10, 20, 30],
    # max_features ikut di-tuning; alasannya diukur di §2.3, bukan diketik di sini.
    "max_features": ["sqrt", None],
}
if FAST_MODE:
    KNN_GRID = {"k": [5, 15], "weights": ["uniform", "distance"], "metric": ["euclidean"]}
    RF_GRID = {"n_estimators": [10, 20], "max_depth": [10, None],
               "max_features": ["sqrt", None]}


### §0.2.1 Kenapa `max_features` masuk grid — dan kenapa `None`

Grid awal hanya menyapu `n_estimators` dan `max_depth`, dengan `max_features`
dibiarkan di nilai bawaan `"sqrt"`. Untuk dataset ini keputusan itu keliru, dan
alasannya bisa dihitung **sebelum** eksperimen dijalankan.

**Mekanismenya.** `d_enc = 60` (dihitung di §2.2), jadi `sqrt(60) = 7`: setiap
simpul hanya mengevaluasi 7 dari 60 kolom. Peluang satu split tidak melihat satu
pun dari tiga kolom geo:

$$\frac{\binom{57}{7}}{\binom{60}{7}} = \frac{53}{60}\cdot\frac{52}{59}\cdot\frac{51}{58} \approx 0{,}685$$

Jadi **±68,5% split tidak pernah melihat kolom geo**, dan terpaksa bercabang di
kolom one-hot yang jauh lebih lemah. Angka MI yang mendasari klaim ini tidak
diketik di sini — dihitung di **§2.3** pada data latih saja, lengkap dengan
tabelnya, supaya tidak bisa basi kalau data berubah.

**Buktinya terukur.** Perbandingan terkontrol pada protokol yang sama
(holdout 0,20, 5-fold, seed 42, `n_estimators=100`, `max_depth=None`):

| `max_features` | CV macro-F1 | test macro-F1 | recall `large` |
| --- | --- | --- | --- |
| `"sqrt"` (7 kolom) | 0,4711 | 0,4737 | 0,0118 |
| `None` (60 kolom) | 0,5165 | 0,5254 | 0,1118 |

Baris `"sqrt"` adalah kombinasi **terbaik dari 12 kombinasi** pada grid lama,
sedangkan baris `None` hanya **satu** konfigurasi. Jadi perbandingan ini justru
menguntungkan `"sqrt"`; selisih **+0,0517** poin macro-F1 dan `recall_large`
yang naik ±9,5× adalah batas bawah, bukan angka optimistis.

**Batas klaimnya.** MI per kolom bukan kebenaran tunggal: satu fitur kategorikal
pecah menjadi beberapa dummy, sehingga MI-nya terbagi dan terlihat kecil
(agregat `landslide_setting` ±0,09 — lihat §2.3). Yang dipakai RF adalah
**kolom**, bukan fitur sumber, jadi granularitas per kolom memang yang relevan.
Klaim yang benar: *geo mendominasi per kolom dan tidak terlihat di 68,5% split* —
bukan *kategorikal tidak berguna*.

Dua konsekuensi sebelum menjalankan ulang:

1. **Biaya.** Grid RF menjadi 24 kombinasi (3 × 4 × 2). `max_features=None`
   mengevaluasi 60 kolom per simpul dan terukur **±4× lebih mahal per pohon**
   daripada `"sqrt"`; estimasi lengkapnya di §A.2/§A.4. Run grid penuh pada
   `N_JOBS=4` memakan ±174 menit.
2. **Cache lama otomatis tidak dipakai.** `_config_signature` (§4) menyertakan
   isi grid, jadi `experiments/random_forest_*.pkl` lama dianggap kedaluwarsa
   dan dihitung ulang. Tidak perlu — dan jangan — menghapus manual: signature
   yang menentukan, bukan nama berkas.

Catatan implementasi: optimasi *prefix pohon* di §4.2 menyimpan hutan dengan
kunci `(semua parameter kecuali n_estimators)`, sehingga hutan `"sqrt"` dan
hutan `None` tidak akan saling tertukar di cache internal fold.

> Riwayat: versi awal §4.2 menyebut "12 kombinasi" dan mengunci `max_features`
> pada `"sqrt"`. Teks itu sudah dikoreksi; jejak keputusan awalnya ada di
> riwayat git, bukan lagi di badan notebook.


### §0.3 Path dan ukuran pekerjaan

Path diturunkan dari lokasi notebook, bukan dari path absolut, supaya tetap jalan setelah
repo dipindah atau di-clone di mesin lain. `assert` pada jumlah CSV mencegah skenario
"melatih model dari dataset yang salah" — lebih baik gagal sekarang daripada menghasilkan
artefak yang tidak bisa dipertanggungjawabkan.

In [4]:
csv_files = list((ROOT / "data").glob("*.csv"))
assert len(csv_files) == 1, f"Harus ada tepat 1 CSV di data/, ditemukan: {csv_files}"
DATA_PATH = csv_files[0]
ARTIFACT_DIR = ROOT / "artifacts" / MODEL_VERSION
EXP_DIR = ROOT / "experiments"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
EXP_DIR.mkdir(exist_ok=True)

n_knn = math.prod(len(v) for v in KNN_GRID.values())
n_rf = math.prod(len(v) for v in RF_GRID.values())
print(f"Dataset    : {DATA_PATH.name}")
print(f"Artefak    : {ARTIFACT_DIR}")
print(f"FAST_MODE  : {FAST_MODE} | FORCE_RERUN: {FORCE_RERUN} | N_JOBS: {N_JOBS}")
print(f"Grid KNN   : {n_knn} kombinasi x {N_SPLITS} fold x {len(TEST_SIZES)} rasio = {n_knn * N_SPLITS * len(TEST_SIZES)} evaluasi CV")
print(f"Grid RF    : {n_rf} kombinasi x {N_SPLITS} fold x {len(TEST_SIZES)} rasio = {n_rf * N_SPLITS * len(TEST_SIZES)} evaluasi CV")

Dataset    : Global_Landslide_Catalog_Export_rows.csv
Artefak    : c:\Kuliah\Semester 7\Penambangan Data dan Analisis\Tugas-Akhir-PDA\backend\artifacts\v1
FAST_MODE  : False | FORCE_RERUN: False | N_JOBS: 4
Grid KNN   : 54 kombinasi x 5 fold x 3 rasio = 810 evaluasi CV
Grid RF    : 24 kombinasi x 5 fold x 3 rasio = 360 evaluasi CV


## §1. Dari data mentah ke data bersih

Data mentah GLC berisi ±11 ribu kejadian longsor global. Sebelum bisa dilatih, ada empat
alasan pembuangan baris yang diterapkan `clean_raw` **secara berurutan**:

1. **Target kosong / `unknown`** — tanpa label tidak ada yang bisa dipelajari.
2. **`event_date` tidak bisa di-parse** — dibutuhkan untuk fitur musiman.
3. **Koordinat di luar rentang** (lat ∉ [-90, 90], lon ∉ [-180, 180]) — fitur geografis
   akan rusak dan jarak KNN jadi tak bermakna.
4. **Duplikat pada `(event_date, latitude, longitude)`** — satu kejadian yang tercatat
   berkali-kali membuat model menghafal titik, bukan belajar pola.

Dua hal yang **tidak** dibuang:

- **Alias target.** `very_large` dan `catastrophic` digabung menjadi `large` sesuai SPEC
  §2.1, sehingga kelas akhirnya tiga: `small`, `medium`, `large`.
- **Nilai kosong pada fitur kategorikal** — diimputasi menjadi kategori `unknown` oleh
  `clean_categorical`. Membuang baris hanya karena satu kolom kosong terlalu mahal untuk
  dataset sekecil ini.

### §1.1 Jalankan pembersihan

Satu panggilan `clean_raw` — sengaja tidak dipecah per langkah di notebook, karena urutan
dan aturannya harus identik dengan yang dipakai backend saat validasi input.

In [5]:
df_raw = pd.read_csv(DATA_PATH)
print(f"Data mentah             : {len(df_raw):,} baris x {df_raw.shape[1]} kolom")

raw_target = df_raw[TARGET].astype("string").str.strip().str.lower()
alias_counts = raw_target.value_counts()
n_alias = int(alias_counts.get("very_large", 0) + alias_counts.get("catastrophic", 0))

df_clean = clean_raw(df_raw)
n_dropped = len(df_raw) - len(df_clean)

print(f"Setelah clean_raw       : {len(df_clean):,} baris")
print(f"Dibuang total           : {n_dropped:,} baris ({n_dropped / len(df_raw) * 100:.1f}%)")
print("  alasan: target kosong/unknown, tanggal invalid, koordinat invalid, duplikat")
print(f"Alias digabung ke large : {n_alias:,} baris (very_large + catastrophic)")

Data mentah             : 11,033 baris x 31 kolom
Setelah clean_raw       : 10,145 baris
Dibuang total           : 888 baris (8.0%)
  alasan: target kosong/unknown, tanggal invalid, koordinat invalid, duplikat
Alias digabung ke large : 105 baris (very_large + catastrophic)


### §1.2 Diagnostik: apa yang diimputasi dan apa yang lolos

Dua hal yang perlu terlihat di sini, bukan diasumsikan:

- berapa nilai kosong pada fitur kategorikal yang **diimputasi** (bukan dibuang), dan
- berapa duplikat yang **masih tersisa** kalau jam diabaikan. `clean_raw` membandingkan
  timestamp lengkap; sisa beberapa baris ini dibiarkan karena jumlahnya sangat kecil dan
  membuangnya berarti mengubah kontrak dataset di luar SPEC. Dicetak supaya keterbatasan
  ini tercatat.

In [22]:
print("Nilai kosong/whitespace pada fitur kategorikal (data mentah; diimputasi, bukan dibuang):")
for col in CATEGORICAL:
    s = df_raw[col].astype("string").str.strip()
    n_missing = int(s.isna().sum() + (s == "").sum())
    print(f"  {col:22s} {n_missing:5,}")

date_only_dups = (
    df_clean.assign(d=df_clean["event_date"].dt.normalize())
    .duplicated(["d", "latitude", "longitude"])
    .sum()
)
print(f"\nDuplikat tersisa bila jam diabaikan: {date_only_dups} baris (dibiarkan, lihat §1)")

Nilai kosong/whitespace pada fitur kategorikal (data mentah; diimputasi, bukan dibuang):
  landslide_category         1
  landslide_trigger         23
  landslide_setting         69
  location_accuracy          2

Duplikat tersisa bila jam diabaikan: 5 baris (dibiarkan, lihat §1)


### §1.3 Sebaran kelas

Ini angka yang menentukan pilihan metrik. Kalau satu kelas mendominasi, akurasi jadi
metrik yang menyesatkan dan macro-F1 yang harus dipakai (§3).

In [23]:
dist = df_clean[TARGET].value_counts().reindex(CLASS_LABELS)
share = (dist / dist.sum() * 100).round(1)
print("Sebaran kelas target:")
print(pd.DataFrame({"jumlah": dist, "persen": share}).to_string())
print(
    f"\nKelas minoritas: '{share.idxmin()}' ({share.min():.1f}%). Karena itu akurasi saja "
    "menyesatkan dan macro-F1 dipakai sebagai metrik utama (§3)."
)

Sebaran kelas target:
                jumlah  persen
landslide_size                
small             2759    27.2
medium            6532    64.4
large              854     8.4

Kelas minoritas: 'large' (8.4%). Karena itu akurasi saja menyesatkan dan macro-F1 dipakai sebagai metrik utama (§3).


## §2. Feature engineering — 9 fitur, tanpa `year`

`engineer_features` menghasilkan **tepat 9 fitur**: 5 numerik + 4 kategorikal.

| Fitur | Asal | Alasan |
| --- | --- | --- |
| `month_sin`, `month_cos` | bulan dari `event_date` | musim bersifat siklikal; pasangan sin/cos membuat Desember dan Januari berdekatan, sesuatu yang mustahil dengan kolom bulan biasa |
| `geo_x`, `geo_y`, `geo_z` | lat/lon → bola 3D | jarak di permukaan bumi tidak linier pada koordinat datar; proyeksi bola membuat kedekatan spasial bermakna — krusial untuk KNN |
| `landslide_category`, `landslide_trigger`, `landslide_setting`, `location_accuracy` | kategorikal mentah | konteks kejadian: jenis longsor, pemicu, lanskap, dan presisi lokasi |

**Kenapa `year` dibuang?** Tahun membawa risiko *temporal leakage* dan *distribution
shift*: model bisa menghafal "longsor besar banyak di tahun X" alih-alih belajar pola
fisik, lalu jeblok pada data tahun berikutnya. Selain itu form operator tidak menjamin
tahun yang konsisten. Keputusan ini tertulis di SPEC §2.1 dan diwujudkan sebagai
`NUMERIC` pada `app/core/constants.py` — satu sumber kebenaran untuk notebook *dan*
backend.

### §2.1 Bentuk data latih

`assert` pada urutan kolom bukan formalitas: urutan kolom adalah bagian dari kontrak.
Kalau `engineer_features` suatu saat mengubah urutan, model yang tersimpan akan menerima
fitur pada posisi yang salah tanpa error — prediksi jadi ngawur tanpa jejak. Lebih baik
gagal di sini.

In [6]:
df_model = engineer_features(df_clean)
if FAST_MODE:
    df_model = df_model.sample(n=2500, random_state=SEED).reset_index(drop=True)

assert list(df_model.columns) == FEATURE_COLUMNS + [TARGET], \
    "urutan kolom menyimpang dari kontrak FEATURE_COLUMNS"
print(f"df_model    : {len(df_model):,} baris x {len(FEATURE_COLUMNS)} fitur (+ target)")
print(f"numerik     : {NUMERIC}")
print(f"kategorikal : {CATEGORICAL}")

df_model    : 10,145 baris x 9 fitur (+ target)
numerik     : ['month_sin', 'month_cos', 'geo_x', 'geo_y', 'geo_z']
kategorikal : ['landslide_category', 'landslide_trigger', 'landslide_setting', 'location_accuracy']


### §2.2 `d_enc` — lebar vektor setelah preprocessing

`Preprocessor` menstandardisasi 5 kolom numerik dan melakukan one-hot pada 4 kolom
kategorikal. Lebar akhirnya disebut **`d_enc`** dan nilainya ditentukan oleh data latih.

Angka ini bukan sekadar informasi: backend memakainya untuk memvalidasi bentuk vektor
input. Kalau preprocessor dan model tidak sinkron, backend menolak start alih-alih
menghasilkan prediksi sampah.

In [7]:
_probe = Preprocessor().fit(df_model)
print(f"Lebar vektor setelah preprocessing (d_enc) = {_probe.n_features_out_}")
print(f"  {'numerik (5 kolom)':24s} -> {len(NUMERIC)} kolom")
for col, cats in zip(CATEGORICAL, _probe.encoder_.categories_):
    print(f"  {col:24s} -> {len(cats)} kolom one-hot")
print(f"  {'total':24s} -> {_probe.n_features_out_} kolom")
print(
    f"\nmax_features RF = floor(sqrt({_probe.n_features_out_})) = "
    f"{int(np.floor(np.sqrt(_probe.n_features_out_)))}"
)

print("\nContoh 3 baris hasil feature engineering:")
print(df_model.head(3).to_string(index=False))

Lebar vektor setelah preprocessing (d_enc) = 60
  numerik (5 kolom)        -> 5 kolom
  landslide_category       -> 14 kolom one-hot
  landslide_trigger        -> 18 kolom one-hot
  landslide_setting        -> 14 kolom one-hot
  location_accuracy        -> 9 kolom one-hot
  total                    -> 60 kolom

max_features RF = floor(sqrt(60)) = 7

Contoh 3 baris hasil feature engineering:
 month_sin  month_cos     geo_x     geo_y     geo_z landslide_category landslide_trigger landslide_setting location_accuracy landslide_size
 -0.866025  -0.500000 -0.252735  0.804018  0.538219          landslide              rain              mine           unknown          large
  0.500000   0.866025 -0.378816 -0.590905  0.712271           mudslide          downpour           unknown               5km          small
  0.500000   0.866025  0.248013 -0.949331 -0.193027          landslide          downpour           unknown              10km          large


### §2.3 Bukti `max_features`: MI dan peluang split buta geo

Sel ini yang membuat argumen §0.2.1 bisa diperiksa, bukan dipercaya. Tiga hal
dihitung, semuanya **pada data latih rasio primer** — baris uji tidak disentuh,
jadi memilih `max_features=None` berdasarkan angka ini tidak mengintip data uji
(aturan yang sama dengan §3). scikit-learn dipakai di sini hanya untuk
mengestimasi MI (diagnostik), bukan untuk melatih model yang diekspor.

1. **MI per fitur mentah** (9 fitur). Kategorikal diwakili kode integer dan
   diperlakukan diskret, sehingga MI-nya dihitung eksak dari tabel kontingensi.
2. **MI per kolom setelah encoding** (60 kolom). Ini satuan yang benar-benar
   dilihat pohon saat memilih split: fitur kontinu memakai estimator k-NN,
   kolom one-hot memakai MI eksak.
3. **Peluang satu split tidak melihat kolom geo**, dihitung dari `d_enc` dan
   `max_features` — aritmetika kombinatorik, bukan simulasi.

Kenapa dibedakan: satu fitur kategorikal pecah menjadi beberapa dummy, jadi MI
per dummy lebih kecil daripada MI variabelnya. Karena `max_features` memilih
**kolom**, angka per kolom yang relevan; angka agregat ditampilkan hanya supaya
pembaca tidak salah menyimpulkan bahwa kategorikal tidak berguna.


In [8]:
# §2.3 — Bukti untuk `max_features`, dihitung ulang di sini.
#
# Semua angka dihitung HANYA pada data latih rasio primer: baris uji tidak
# disentuh, jadi memilih `max_features=None` berdasarkan angka ini tidak
# mengintip data uji (aturan yang sama dengan §3).
from math import comb

from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import mutual_info_score

y_mi = encode_target(df_model[TARGET].to_numpy())
tr_mi, _, y_tr_mi, _ = stratified_holdout(df_model, y_mi, PRIMARY_TEST_SIZE, SEED)
prep_mi = Preprocessor().fit(tr_mi)
X_mi = prep_mi.transform(tr_mi)

enc_names = list(NUMERIC)
for col, cats in zip(CATEGORICAL, prep_mi.encoder_.categories_):
    enc_names += [f"{col}={cat}" for cat in cats]
assert len(enc_names) == X_mi.shape[1] == prep_mi.n_features_out_


def _mi(column, discrete):
    """MI terhadap target. Diskret -> mutual_info_score, kontinu -> estimator k-NN.

    Ini persis dua jalur yang dipakai ``mutual_info_classif``; dipanggil eksplisit
    agar tidak memunculkan peringatan dtype palsu dari ``check_clusterings``.
    """
    values = np.asarray(column)
    if discrete:
        return float(mutual_info_score(values.astype(np.int64), y_tr_mi))
    return float(
        mutual_info_classif(
            values.astype(np.float64).reshape(-1, 1),
            y_tr_mi,
            discrete_features=False,
            random_state=SEED,
        )[0]
    )


# (a) MI per fitur mentah (9 fitur). Kategorikal = kode integer, jadi diskret.
raw = tr_mi[NUMERIC + CATEGORICAL].copy()
for col in CATEGORICAL:
    raw[col] = raw[col].astype("category").cat.codes
raw_mi = {col: _mi(raw[col].to_numpy(), col in CATEGORICAL) for col in raw.columns}

print(f"(a) MI per fitur mentah — data latih {len(tr_mi):,} baris:")
for name, value in sorted(raw_mi.items(), key=lambda kv: kv[1], reverse=True):
    print(f"    {name:22s} {value:.4f}")

# (b) Satuan yang benar-benar dilihat pohon: 60 kolom hasil encoding.
n_numeric = len(NUMERIC)
col_mi = np.array([_mi(X_mi[:, i], i >= n_numeric) for i in range(X_mi.shape[1])])
geo = {"geo_x", "geo_y", "geo_z"}
geo_idx = [i for i, name in enumerate(enc_names) if name in geo]
non_geo = [(name, value) for name, value in zip(enc_names, col_mi) if name not in geo]
best_non_geo = max(non_geo, key=lambda kv: kv[1])

print(f"\n(b) MI per kolom setelah encoding ({X_mi.shape[1]} kolom) — top 6:")
for i in np.argsort(col_mi)[::-1][:6]:
    print(f"    {enc_names[i]:32s} {col_mi[i]:.4f}")
print(f"    {'kolom non-geo tertinggi':32s} {best_non_geo[1]:.4f}  ({best_non_geo[0]})")

# Satu fitur kategorikal pecah jadi beberapa dummy; MI-nya harus dijumlahkan
# sebelum dibandingkan dengan fitur numerik. RF memilih KOLOM, bukan fitur
# sumber, jadi angka per kolom di atas yang relevan untuk max_features.
agg = {name: 0.0 for name in NUMERIC + CATEGORICAL}
for name, value in zip(enc_names, col_mi):
    agg[name.split("=")[0]] += float(value)
print("\n    agregat per fitur sumber (jumlah MI semua dummy):")
for name, value in sorted(agg.items(), key=lambda kv: kv[1], reverse=True):
    print(f"      {name:22s} {value:.4f}")

# (c) Peluang satu split tidak melihat kolom geo mana pun.
n_enc = X_mi.shape[1]
k_sqrt = max(1, int(np.floor(np.sqrt(n_enc))))
n_geo = len(geo_idx)
p_blind = comb(n_enc - n_geo, k_sqrt) / comb(n_enc, k_sqrt)

print("\n(c) Peluang satu split tidak melihat kolom geo mana pun:")
print(f"    d_enc={n_enc}, 'sqrt' -> {k_sqrt} kolom/split, {n_geo} kolom geo")
print(f"    P(buta geo)        = C({n_enc - n_geo},{k_sqrt}) / C({n_enc},{k_sqrt}) = {p_blind:.4f}")
print(f"    P(melihat >=1 geo) = {1 - p_blind:.4f}")
print(f"    'None' -> {n_enc} kolom/split, P(buta geo) = 0.0000")

# (d) Kesimpulan yang bisa dibaca tanpa membuka markdown.
geo_mi = col_mi[geo_idx]
print("\nKesimpulan:")
print(f"  - Tiga kolom geo punya MI {geo_mi.min():.3f}-{geo_mi.max():.3f}, konsisten jadi yang tertinggi.")
print(f"  - Dengan 'sqrt', {p_blind:.1%} split tidak pernah melihat satu pun kolom itu.")
print(f"  - Split yang buta geo terpaksa bercabang di kolom non-geo terbaik (MI {best_non_geo[1]:.3f}).")
print("  - Karena itu max_features ikut di-tuning, bukan dikunci; §4.2 menguji hasilnya di data uji.")


(a) MI per fitur mentah — data latih 8,118 baris:
    geo_y                  0.1178
    geo_z                  0.1073
    geo_x                  0.0679
    location_accuracy      0.0610
    landslide_setting      0.0575
    month_sin              0.0308
    landslide_category     0.0279
    landslide_trigger      0.0246
    month_cos              0.0000

(b) MI per kolom setelah encoding (60 kolom) — top 6:
    geo_y                            0.1178
    geo_z                            0.1074
    geo_x                            0.0679
    landslide_setting=unknown        0.0424
    landslide_setting=above_road     0.0381
    month_sin                        0.0308
    kolom non-geo tertinggi          0.0424  (landslide_setting=unknown)

    agregat per fitur sumber (jumlah MI semua dummy):
      geo_y                  0.1178
      geo_z                  0.1074
      landslide_setting      0.0921
      location_accuracy      0.0704
      geo_x                  0.0679
      landslide_c

## §3. Desain evaluasi: kenapa tidak ada leakage

Aturannya satu: **apa pun yang "belajar dari data" hanya boleh melihat data latih.**

Alur per rasio holdout:

1. `stratified_holdout` membagi data menjadi train/test dengan proporsi kelas yang sama.
   Stratifikasi wajib karena `large` hanya ±8%; split acak biasa bisa membuat kelas
   minoritas nyaris hilang dari data uji, dan metriknya jadi tidak bisa dipercaya.
2. Di dalam data train, `holdout_grid_search` menjalankan 5-fold CV. **Preprocessor
   di-fit ulang di setiap fold, hanya pada bagian train fold itu.** Statistik scaler dan
   kosakata one-hot tidak pernah "mengintip" fold validasi.
3. Kombinasi hyperparameter terbaik di-refit pada 100% data train, lalu diuji **sekali**
   pada test set yang sejak awal tidak pernah disentuh.

Konsekuensi yang sering terlewat: `d_enc` di dalam CV bisa lebih kecil daripada `d_enc`
final, karena kategori langka yang hanya muncul di sebagian data mungkin tidak ada di
fold tertentu. Itu bukan bug — itu bukti refit per fold benar-benar terjadi.

**Metrik utama: macro-F1.** Akurasi menyesatkan pada data tidak seimbang: menebak
`medium` untuk semua baris sudah memberi akurasi ±64%. Macro-F1 memberi bobot sama ke
tiap kelas sehingga `large` yang jarang tetap diperhitungkan. `recall_large` dilaporkan
terpisah karena secara operasional justru kelas itu yang paling penting — longsor besar
yang salah diprediksi sebagai kecil adalah kesalahan yang mahal.

**Kenapa CV *dan* holdout?** CV dipakai untuk *memilih* hyperparameter (lebih stabil
daripada satu split). Holdout dipakai untuk *melaporkan* performa (benar-benar data yang
tidak terlibat pemilihan). Memilih model berdasarkan skor holdout lalu melaporkan skor
yang sama adalah cara klasik menghasilkan angka optimistis.

### §3.1 Holdout — ukuran dan proporsi kelas

Yang dicek: proporsi kelas di data uji harus mendekati sebaran penuh. Kalau `large` hilang
dari data uji, `recall_large` di §5 tidak bisa dipercaya.

In [27]:
y_demo = encode_target(df_model[TARGET].to_numpy())


def _class_shares(y):
    return "  ".join(f"{(y == i).mean() * 100:5.1f}%" for i in range(len(CLASS_LABELS)))


print("Holdout: ukuran dan proporsi kelas di data UJI")
print("rasio  n_train  n_test   " + "  ".join(f"{c:>6s}" for c in CLASS_LABELS))
for ratio in TEST_SIZES:
    tr, te, ytr, yte = stratified_holdout(df_model, y_demo, ratio, SEED)
    print(f"{ratio:5.2f}  {len(tr):7,}  {len(te):6,}   {_class_shares(yte)}")

Holdout: ukuran dan proporsi kelas di data UJI
rasio  n_train  n_test    small  medium   large
 0.10    9,132   1,013    27.1%   64.5%    8.4%
 0.15    8,625   1,520    27.2%   64.4%    8.4%
 0.20    8,118   2,027    27.2%   64.4%    8.4%


### §3.2 K-Fold di dalam data latih

CV dijalankan **hanya** pada data latih rasio primer. Perhatikan bahwa setiap sampel jadi
data validasi tepat satu kali — itu definisi K-Fold yang benar; kalau ada yang muncul dua
kali, estimasi CV-nya bias.

In [28]:
tr_p, te_p, ytr_p, yte_p = stratified_holdout(df_model, y_demo, PRIMARY_TEST_SIZE, SEED)
print(f"5-Fold pada data latih rasio primer ({PRIMARY_TEST_SIZE:.0%}, {len(tr_p):,} baris):")
folds = stratified_kfold(ytr_p, N_SPLITS, SEED)
for i, (tr_idx, va_idx) in enumerate(folds, 1):
    print(f"  fold {i}: latih {len(tr_idx):6,} | validasi {len(va_idx):5,}   kelas validasi: {_class_shares(ytr_p[va_idx])}")

tervalidasi = np.concatenate([va for _, va in folds])
assert len(tervalidasi) == len(ytr_p) and len(np.unique(tervalidasi)) == len(ytr_p)
print("OK: setiap sampel menjadi validasi tepat satu kali")

5-Fold pada data latih rasio primer (20%, 8,118 baris):
  fold 1: latih  6,493 | validasi 1,625   kelas validasi:  27.2%   64.4%    8.4%
  fold 2: latih  6,494 | validasi 1,624   kelas validasi:  27.2%   64.3%    8.4%
  fold 3: latih  6,494 | validasi 1,624   kelas validasi:  27.2%   64.3%    8.4%
  fold 4: latih  6,495 | validasi 1,623   kelas validasi:  27.2%   64.4%    8.4%
  fold 5: latih  6,496 | validasi 1,622   kelas validasi:  27.2%   64.4%    8.4%
OK: setiap sampel menjadi validasi tepat satu kali


## §4. Menjalankan eksperimen (dengan cache)

Tiga helper kecil mengurus seluruh §4:

- **`_config_signature`** — sidik jari konfigurasi. Isinya sengaja mencakup `n_rows`,
  bukan hanya hyperparameter: mengganti dataset tanpa mengubah grid juga harus
  membatalkan cache.
- **`run_or_load`** — untuk setiap rasio: pakai cache bila signature cocok, kalau tidak
  jalankan `holdout_grid_search` lalu simpan. Penulisan memakai berkas `.tmp` +
  `replace()` supaya proses yang mati di tengah tidak meninggalkan cache setengah jadi.
- **`summarize` / `show_summary`** — merapikan hasil jadi tabel yang bisa dibaca.

Progress bar menghitung **langkah**, bukan detik: satu rasio = `N_SPLITS × jumlah kombinasi + 1`
(langkah terakhir adalah refit + uji). Jadi kalau bar terlihat macet, kelihatan bagian mana
yang sedang dikerjakan.

In [29]:
def _config_signature(grid):
    # Identitas eksperimen. Jika berubah, hasil tersimpan dianggap kedaluwarsa.
    return {
        "grid": {k: [str(v) for v in vals] for k, vals in grid.items()},
        "seed": SEED,
        "n_splits": N_SPLITS,
        "n_rows": int(len(df_model)),
        "fast_mode": FAST_MODE,
    }

Cache disimpan per (algoritma, rasio). Yang membuat cache aman dipakai adalah pemeriksaan
signature di bawah: berkas yang ada tapi signature-nya beda **dijalankan ulang**, bukan
dipercaya. Penulisan memakai `.tmp` + `replace()` supaya proses yang mati di tengah tidak
meninggalkan berkas setengah jadi yang lolos validasi.

In [30]:
def run_or_load(name, factory, grid):
    sig = _config_signature(grid)
    n_combo = math.prod(len(v) for v in grid.values())
    steps_per_ratio = N_SPLITS * n_combo + 1          # +1 = refit & uji

    def ckpt_path(ratio):
        tag = "_fast" if FAST_MODE else ""
        return EXP_DIR / f"{name}_{int(round(ratio * 100)):02d}{tag}.pkl"

    reports = []
    t0 = time.time()
    with tqdm(total=len(TEST_SIZES) * steps_per_ratio, desc=name, unit="langkah") as bar:
        for ratio in TEST_SIZES:
            path = ckpt_path(ratio)

            # 1) pakai checkpoint bila ada dan konfigurasinya sama
            if path.exists() and not FORCE_RERUN:
                with open(path, "rb") as f:
                    saved = pickle.load(f)
                if saved.get("signature") == sig:
                    reports.append(saved["report"])
                    bar.update(steps_per_ratio)
                    bar.set_postfix_str(f"rasio {ratio:.0%}: dimuat")
                    continue
                print(f"[{name}] {path.name}: konfigurasi berubah, dijalankan ulang")

            # 2) jalankan satu rasio, lalu langsung simpan
            t_ratio = time.time()
            report = holdout_grid_search(
                factory, grid, df_model,
                test_sizes=(ratio,), n_splits=N_SPLITS, seed=SEED,
                on_step=bar.update,
            )[0]
            tmp = path.with_suffix(".tmp")
            with open(tmp, "wb") as f:
                pickle.dump({"signature": sig, "report": report}, f)
            tmp.replace(path)                         # tulis atomik, anti file setengah jadi
            reports.append(report)
            bar.set_postfix_str(f"rasio {ratio:.0%}: {(time.time() - t_ratio) / 60:.1f} mnt")

    print(f"[{name}] selesai, total {(time.time() - t0) / 60:.1f} menit")
    return reports

`summarize` menyusun satu baris per (algoritma, rasio). `show_summary` mencetaknya plus
selisih `|CV − uji|`: kalau selisihnya besar, artinya hyperparameter yang menang di CV
tidak terbukti di data uji — tanda overfitting pada proses tuning.

In [31]:
def summarize(name, reports):
    rows = []
    for r in reports:
        m = r["test_metrics"]
        rows.append({
            "algoritma": name,
            "test_size": r["test_size"],
            "n_train": r["n_train"],
            "n_test": r["n_test"],
            "cv_macro_f1": round(r["cv_mean_macro_f1"], 4),
            "cv_std": round(r["cv_std_macro_f1"], 4),
            "test_macro_f1": round(m["macro_f1"], 4),
            "test_bal_acc": round(m["balanced_accuracy"], 4),
            "test_acc": round(m["accuracy"], 4),
            "recall_large": round(m["per_class"]["large"]["recall"], 4),
            "best_params": r["best_params"],
        })
    return pd.DataFrame(rows)


def show_summary(df, title):
    cols = ["test_size", "n_train", "n_test", "cv_macro_f1", "cv_std",
            "test_macro_f1", "test_bal_acc", "recall_large", "best_params"]
    print(title)
    print(df[cols].to_string(index=False))
    gap = (df["cv_macro_f1"] - df["test_macro_f1"]).abs()
    print(f"|CV - uji| per rasio: {', '.join(f'{v:.4f}' for v in gap)}")
    print("Selisih kecil berarti hyperparameter pilihan CV tidak menipu di data uji.")

### §4.1 KNN — 54 kombinasi

KNN tidak punya "model" yang dilatih; ia menyimpan data latih dan memutuskan berdasarkan
tetangga terdekat. Yang bisa dioptimalkan hanya cara mengukur kedekatan dan cara memberi
suara:

| Hyperparameter | Nilai | Kenapa |
| --- | --- | --- |
| `k` | 3, 5, 7, 9, 11, 15, 21, 31, 41 | dari sangat lokal (kecil, varians tinggi) sampai sangat halus (besar, bias tinggi) |
| `weights` | `uniform`, `distance` | `distance` memberi tetangga dekat suara lebih besar — biasanya menolong bila kepadatan data tidak merata |
| `metric` | `euclidean`, `manhattan`, `chebyshev` | 9 × 2 × 3 = **54 kombinasi** |

Fitur kategorikal sudah di-one-hot dan numerik sudah distandardisasi, jadi skala antar
fitur setara dan jarak punya arti. Tanpa langkah itu, KNN akan didominasi kolom dengan
rentang terbesar — kesalahan klasik yang membuat hasil KNN tampak buruk padahal
masalahnya di preprocessing.

Implementasi manual menghitung matriks jarak **sekali per fold per metric**, lalu
memakainya ulang untuk semua kombinasi `k` dan `weights`. Hasilnya identik dengan cara
naif, tetapi jauh lebih cepat.

In [32]:
knn_reports = run_or_load("knn", lambda **p: KNNClassifier(**p), KNN_GRID)
knn_summary = summarize("KNN", knn_reports)
show_summary(knn_summary, "Ringkasan KNN per rasio holdout:")

knn:   0%|          | 0/813 [00:00<?, ?langkah/s]

[knn] selesai, total 0.0 menit
Ringkasan KNN per rasio holdout:
 test_size  n_train  n_test  cv_macro_f1  cv_std  test_macro_f1  test_bal_acc  recall_large                                            best_params
      0.10     9132    1013       0.5163  0.0136         0.5165        0.5065        0.1294 {'k': 7, 'weights': 'distance', 'metric': 'manhattan'}
      0.15     8625    1520       0.5145  0.0041         0.5077        0.4978        0.1562 {'k': 5, 'weights': 'distance', 'metric': 'manhattan'}
      0.20     8118    2027       0.5124  0.0136         0.5013        0.4924        0.1353 {'k': 5, 'weights': 'distance', 'metric': 'manhattan'}
|CV - uji| per rasio: 0.0002, 0.0068, 0.0111
Selisih kecil berarti hyperparameter pilihan CV tidak menipu di data uji.


### §4.2 Random Forest — 24 kombinasi + optimasi prefix pohon

| Hyperparameter | Nilai | Kenapa |
| --- | --- | --- |
| `n_estimators` | 100, 200, 300 | lebih banyak pohon menurunkan varians, tapi biaya naik linier |
| `max_depth` | `None`, 10, 20, 30 | `None` = pohon tumbuh penuh (bias rendah, varians tinggi); batas kedalaman mengontrol overfitting |
| `max_features` | `"sqrt"`, `None` | `sqrt(60)=7` membuat ±68,5% split buta terhadap kolom geo; `None` mengevaluasi semua 60 kolom. Bukti dan biayanya di §0.2.1 dan §2.3 |

3 × 4 × 2 = **24 kombinasi**. Melatih 300 pohon untuk setiap kombinasi berarti
24 × 300 pohon per fold. Karena pohon ke-`n` dari hutan yang sama tidak berubah
ketika `n_estimators` bertambah (hanya jumlah pohonnya), implementasi manual
melatih **8 hutan** 300 pohon — satu per pasangan (`max_depth`, `max_features`) —
**sekali per fold**, lalu mengevaluasi 100/200/300 pohon pertama. Hasilnya
identik dengan melatih ulang, biayanya sepertiga.


In [33]:
rf_reports = run_or_load(
    "random_forest",
    lambda **p: RandomForest(random_state=SEED, n_jobs=N_JOBS, **p),
    RF_GRID,
)
rf_summary = summarize("Random Forest", rf_reports)
show_summary(rf_summary, "Ringkasan Random Forest per rasio holdout:")

random_forest:   0%|          | 0/363 [00:00<?, ?langkah/s]

[random_forest] random_forest_10.pkl: konfigurasi berubah, dijalankan ulang
[random_forest] random_forest_15.pkl: konfigurasi berubah, dijalankan ulang
[random_forest] random_forest_20.pkl: konfigurasi berubah, dijalankan ulang
[random_forest] selesai, total 174.0 menit
Ringkasan Random Forest per rasio holdout:
 test_size  n_train  n_test  cv_macro_f1  cv_std  test_macro_f1  test_bal_acc  recall_large                                                    best_params
      0.10     9132    1013       0.5216  0.0097         0.5402        0.5240        0.1294   {'n_estimators': 200, 'max_depth': 30, 'max_features': None}
      0.15     8625    1520       0.5239  0.0151         0.5355        0.5229        0.1094 {'n_estimators': 300, 'max_depth': None, 'max_features': None}
      0.20     8118    2027       0.5195  0.0190         0.5327        0.5195        0.1118   {'n_estimators': 300, 'max_depth': 20, 'max_features': None}
|CV - uji| per rasio: 0.0186, 0.0116, 0.0132
Selisih kecil berarti

## §5. Membandingkan dan memilih model

Aturan pemilihan dibuat eksplisit supaya tidak bisa "disesuaikan setelah melihat hasil":

1. Untuk setiap algoritma, ambil `test_macro_f1` di **ketiga** rasio holdout.
2. Pemenang = rata-rata tertinggi. Rata-rata, bukan skor terbaik: memilih berdasarkan
   satu rasio berarti memilih berdasarkan keberuntungan split.
3. Runner-up disimpan sebagai **comparator**, bukan dibuang — laporan butuh angka
   pembanding, dan SPEC §2.4 meminta berkasnya ada.
4. Parameter yang diekspor diambil dari run **rasio primer** (0.20), bukan dari rasio yang
   kebetulan memberi skor tertinggi.

### §5.1 Tabel gabungan

Enam baris, satu per (algoritma, rasio). Dibaca berpasangan: bandingkan baris KNN dan RF
pada rasio yang sama, jangan membandingkan rasio yang berbeda.

In [34]:
summary = pd.concat([knn_summary, rf_summary], ignore_index=True)
display_cols = ["algoritma", "test_size", "cv_macro_f1", "cv_std", "test_macro_f1",
                "test_bal_acc", "test_acc", "recall_large"]
print("Seluruh hasil eksperimen:")
print(summary[display_cols].to_string(index=False))

mean_f1 = summary.groupby("algoritma")["test_macro_f1"].mean().sort_values(ascending=False)
print("\nRata-rata macro-F1 uji di 3 rasio:")
for name, val in mean_f1.items():
    print(f"  {name:15s} {val:.4f}")

Seluruh hasil eksperimen:
    algoritma  test_size  cv_macro_f1  cv_std  test_macro_f1  test_bal_acc  test_acc  recall_large
          KNN       0.10       0.5163  0.0136         0.5165        0.5065    0.6940        0.1294
          KNN       0.15       0.5145  0.0041         0.5077        0.4978    0.6724        0.1562
          KNN       0.20       0.5124  0.0136         0.5013        0.4924    0.6759        0.1353
Random Forest       0.10       0.5216  0.0097         0.5402        0.5240    0.7196        0.1294
Random Forest       0.15       0.5239  0.0151         0.5355        0.5229    0.7237        0.1094
Random Forest       0.20       0.5195  0.0190         0.5327        0.5195    0.7218        0.1118

Rata-rata macro-F1 uji di 3 rasio:
  Random Forest   0.5361
  KNN             0.5085


### §5.2 Pemenang dan kestabilannya

Selisih rata-rata saja tidak cukup: kalau pemenang hanya unggul di satu rasio, kesimpulan
"model A lebih baik" sebenarnya tidak didukung data. Karena itu dicek juga apakah ia
menang di **semua** rasio.

In [35]:
winner, loser = mean_f1.index[0], mean_f1.index[1]
print(f"Model aktif : {winner} (rata-rata macro-F1 {mean_f1[winner]:.4f})")
print(f"Comparator  : {loser} (rata-rata macro-F1 {mean_f1[loser]:.4f})")
print(f"Selisih     : {mean_f1[winner] - mean_f1[loser]:+.4f} poin")

pivot = summary.pivot(index="test_size", columns="algoritma", values="test_macro_f1")
menang_semua = bool((pivot[winner] > pivot[loser]).all())
print("\nmacro-F1 uji per rasio:")
print(pivot.round(4).to_string())
print("Konsisten menang di semua rasio?", "ya" if menang_semua else "TIDAK — kesimpulan rapuh")

reports_by_name = {"KNN": knn_reports, "Random Forest": rf_reports}
primary = next(r for r in reports_by_name[winner]
               if abs(r["test_size"] - PRIMARY_TEST_SIZE) < 1e-9)
print(f"\nParameter terbaik pada rasio primer ({PRIMARY_TEST_SIZE:.0%}): {primary['best_params']}")

Model aktif : Random Forest (rata-rata macro-F1 0.5361)
Comparator  : KNN (rata-rata macro-F1 0.5085)
Selisih     : +0.0276 poin

macro-F1 uji per rasio:
algoritma     KNN  Random Forest
test_size                       
0.10       0.5165         0.5402
0.15       0.5077         0.5355
0.20       0.5013         0.5327
Konsisten menang di semua rasio? ya

Parameter terbaik pada rasio primer (20%): {'n_estimators': 300, 'max_depth': 20, 'max_features': None}


### §5.3 Pola kesalahan, bukan cuma angka besar

Yang dicari di confusion matrix bukan akurasinya, tapi ke mana kesalahan pergi. Untuk
peringatan dini, kesalahan paling mahal adalah `large` yang diprediksi `small` — model
memberi rasa aman yang salah.

In [36]:
cm = pd.DataFrame(primary["test_metrics"]["confusion_matrix"],
                  index=CLASS_LABELS, columns=CLASS_LABELS)
print("Confusion matrix (baris = aktual, kolom = prediksi):")
print(cm.to_string())

print("\nMetrik per kelas:")
print(pd.DataFrame(primary["test_metrics"]["per_class"]).T.round(4).to_string())

n_large = int(cm.loc["large"].sum())
n_large_hit = int(cm.loc["large", "large"])
baseline = cm.sum(axis=1).max() / cm.values.sum()
print(
    f"\nInterpretasi: dari {n_large} longsor 'large' di data uji, {n_large_hit} terdeteksi "
    f"({n_large_hit / n_large * 100:.1f}%); {n_large - n_large_hit} turun kelas."
)
print(
    f"Akurasi {primary['test_metrics']['accuracy']:.4f} vs baseline tebak-kelas-mayoritas "
    f"{baseline:.4f}: selisihnya tipis, jadi akurasi memang bukan metrik utama di sini."
)

Confusion matrix (baris = aktual, kolom = prediksi):
        small  medium  large
small     325     221      5
medium    167    1119     20
large       9     142     19

Metrik per kelas:
        precision  recall      f1  support
small      0.6487  0.5898  0.6179    551.0
medium     0.7551  0.8568  0.8027   1306.0
large      0.4318  0.1118  0.1776    170.0

Interpretasi: dari 170 longsor 'large' di data uji, 19 terdeteksi (11.2%); 151 turun kelas.
Akurasi 0.7218 vs baseline tebak-kelas-mayoritas 0.6443: selisihnya tipis, jadi akurasi memang bukan metrik utama di sini.


## §6. Ekspor artefak — kontrak ke backend

Empat berkas ditulis ke `artifacts/<versi>/`. Poin pentingnya: backend **tidak** mengimpor
notebook ini dan tidak tahu apa pun soal training. Ia hanya membaca `model_meta.json`,
lalu memuat berkas yang ditunjuk `artifact_files.model`. Karena itu isi meta adalah
kontrak, bukan dokumentasi:

| Kunci | Dipakai backend untuk |
| --- | --- |
| `model_type` | memilih kelas (`KNNClassifier` vs `RandomForest`) saat memuat `.npz` |
| `best_params` | rekonstruksi hyperparameter |
| `d_enc` | memvalidasi lebar vektor hasil preprocessing (fail fast bila preprocessor dan model tidak sinkron) |
| `class_labels` | urutan kolom `predict_proba` |
| `valid_categories` | validasi input operator dan isi `/meta/options` |
| `evaluation_metrics` | ditampilkan di halaman model |

Model pemenang **dan** pembanding sama-sama diekspor. Alasannya literal: SPEC §2.4
meminta kedua berkas model ada. Berkas mana yang aktif ditentukan `model_meta.json`,
jadi mengekspor keduanya tidak menimbulkan ambiguitas.

### §6.1 Latih ulang model pemenang

Model dilatih ulang dari nol pada data train rasio primer. Hasilnya di-`assert` harus sama
dengan hasil eksperimen. Kalau berbeda, ada nondeterminisme yang harus diselidiki
**sebelum** artefak dipakai — bukan setelah prediksi produksi terlihat aneh.

In [37]:
factories = {
    "KNN": ("knn", lambda **p: KNNClassifier(**p)),
    "Random Forest": ("random_forest",
                      lambda **p: RandomForest(random_state=SEED, n_jobs=N_JOBS, **p)),
}
model_type, factory = factories[winner]
best_params = primary["best_params"]

y_all = encode_target(df_model[TARGET].to_numpy())
tr_df, te_df, y_tr, y_te = stratified_holdout(df_model, y_all, PRIMARY_TEST_SIZE, SEED)
prep = Preprocessor()
X_tr = prep.fit_transform(tr_df)
X_te = prep.transform(te_df)
final_model = factory(**best_params).fit(X_tr, y_tr)
final_metrics = evaluate(y_te, final_model.predict(X_te))

assert abs(final_metrics["macro_f1"] - primary["test_metrics"]["macro_f1"]) < 1e-9
print(f"Latih ulang pada {len(tr_df):,} baris -> macro-F1 uji {final_metrics['macro_f1']:.4f} "
      f"(identik dengan hasil eksperimen)")

Latih ulang pada 8,118 baris -> macro-F1 uji 0.5327 (identik dengan hasil eksperimen)


### §6.2 Preprocessor dan model pemenang

Preprocessor disimpan **sebelum** model karena keduanya harus berasal dari objek yang
sama. Kalau suatu saat salah satu di-refit terpisah, `d_enc` bisa berbeda dan backend
menolak start — deteksi dini yang memang diinginkan.

In [38]:
with open(ARTIFACT_DIR / "preprocessor.json", "w", encoding="utf-8") as f:
    json.dump(prep.to_dict(), f)

model_file = "rf_trees.npz" if model_type == "random_forest" else "knn_store.npz"
np.savez_compressed(ARTIFACT_DIR / model_file, **final_model.to_arrays())
print(f"preprocessor.json    : d_enc={prep.n_features_out_}")
print(f"{model_file:22s}: {winner} {best_params}")

preprocessor.json    : d_enc=60
rf_trees.npz          : Random Forest {'n_estimators': 300, 'max_depth': 20, 'max_features': None}


### §6.3 Comparator (runner-up)

Bukan sekadar formalitas SPEC: menyimpan model pembanding membuat klaim "model A lebih
baik dari B" bisa direproduksi tanpa melatih ulang apa pun.

In [39]:
loser_type, loser_factory = factories[loser]
loser_report = next(r for r in reports_by_name[loser]
                    if abs(r["test_size"] - PRIMARY_TEST_SIZE) < 1e-9)
loser_params = loser_report["best_params"]
loser_model = loser_factory(**loser_params).fit(X_tr, y_tr)
loser_file = "rf_trees.npz" if loser_type == "random_forest" else "knn_store.npz"
assert loser_file != model_file, "pemenang dan pembanding tidak boleh sama"
np.savez_compressed(ARTIFACT_DIR / loser_file, **loser_model.to_arrays())
print(f"{loser_file:22s}: {loser} {loser_params}")

knn_store.npz         : KNN {'k': 5, 'weights': 'distance', 'metric': 'manhattan'}


### §6.4 `model_meta.json`

Ini berkas yang benar-benar menentukan perilaku backend. Yang perlu diperhatikan: tidak
ada nilai yang diketik manual di sini — semuanya berasal dari objek yang baru dilatih dan
dari `summary`, sehingga meta tidak bisa "berbohong" soal performa.

In [40]:
meta = {
    "model_type": model_type,
    "model_name": winner,
    "version": MODEL_VERSION,
    "trained_at": datetime.now(timezone.utc).isoformat(),
    "best_params": best_params,
    "seed": SEED,
    "holdout_ratio": PRIMARY_TEST_SIZE,
    "n_train": int(len(tr_df)),
    "n_test": int(len(te_df)),
    "feature_names": FEATURE_COLUMNS,
    "numeric_features": NUMERIC,
    "categorical_features": CATEGORICAL,
    "d_enc": int(prep.n_features_out_),
    "class_labels": CLASS_LABELS,
    "valid_categories": {c: sorted(df_model[c].unique().tolist()) for c in CATEGORICAL},
    "coordinate_bounds": {"latitude": [-90.0, 90.0], "longitude": [-180.0, 180.0]},
    "evaluation_metrics": final_metrics,
    "comparator_info": {
        "model_name": loser,
        "best_params": loser_params,
        "mean_test_macro_f1": round(float(mean_f1[loser]), 4),
        "artifact_file": loser_file,
    },
    "experiment_summary": summary.to_dict("records"),
    "artifact_files": {
        "preprocessor": "preprocessor.json",
        "model": model_file,
        "comparator": loser_file,
    },
}
with open(ARTIFACT_DIR / "model_meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

print(f"model_meta.json      : model_type={model_type}, d_enc={meta['d_enc']}, "
      f"model={model_file}")
print("\nIsi folder artefak:")
for p in sorted(ARTIFACT_DIR.iterdir()):
    print(f"  {p.name:22s} {p.stat().st_size / 1024:9.1f} KB")

model_meta.json      : model_type=random_forest, d_enc=60, model=rf_trees.npz

Isi folder artefak:
  knn_store.npz              162.7 KB
  model_meta.json              6.1 KB
  preprocessor.json            1.2 KB
  rf_trees.npz              7646.4 KB


## §7. Verifikasi artefak

Empat pemeriksaan yang membuktikan artefak benar-benar bisa dipakai, bukan sekadar
berhasil ditulis. Semuanya memakai `assert`, jadi kegagalan menghentikan notebook dengan
keras — itu memang tujuannya.

### §7.1 Round-trip: muat ulang dari disk

Menangkap bug serialisasi — kelas bug yang tidak akan terlihat sampai backend melayani
request.

In [41]:
with open(ARTIFACT_DIR / "preprocessor.json", encoding="utf-8") as f:
    prep2 = Preprocessor.from_dict(json.load(f))
with np.load(ARTIFACT_DIR / model_file) as arrays:
    cls = RandomForest if model_type == "random_forest" else KNNClassifier
    model2 = cls.from_arrays(arrays)

X_te2 = prep2.transform(te_df)
assert np.allclose(X_te, X_te2, atol=1e-12)
assert np.array_equal(final_model.predict(X_te), model2.predict(X_te2))
print("OK 1: artefak yang dimuat ulang memberi prediksi identik")

OK 1: artefak yang dimuat ulang memberi prediksi identik


### §7.2 Parity jalur inference vs jalur training

Bug paling umum dan paling sulit dilacak di produksi: urutan kolom berbeda antara
`payload_to_frame` dan `engineer_features`. Di sini keduanya dibandingkan untuk baris yang
sama.

In [42]:
row = df_clean.iloc[0]
payload = {
    "event_date": pd.Timestamp(row["event_date"]).strftime("%Y-%m-%d"),
    "latitude": float(row["latitude"]),
    "longitude": float(row["longitude"]),
    **{c: str(row[c]) for c in CATEGORICAL},
}
v_inference = prep2.transform_one(payload_to_frame(payload))
v_training = prep2.transform(engineer_features(df_clean.iloc[[0]]))[0]
assert np.allclose(v_inference, v_training, atol=1e-12)
print("OK 2: payload_to_frame + transform_one == jalur training")

OK 2: payload_to_frame + transform_one == jalur training


### §7.3 Probabilitas valid

Backend memakai `predict_proba` untuk menampilkan tingkat keyakinan. Kalau jumlahnya bukan
1, tampilan di UI akan menyesatkan.

In [43]:
proba = model2.predict_proba(v_inference.reshape(1, -1))[0]
assert proba.shape == (len(CLASS_LABELS),)
assert np.all(proba >= 0) and abs(float(proba.sum()) - 1.0) < 1e-9
print("OK 3: probabilitas", dict(zip(CLASS_LABELS, proba.round(3))),
      "| jumlah =", round(float(proba.sum()), 6))

OK 3: probabilitas {'small': np.float64(0.104), 'medium': np.float64(0.22), 'large': np.float64(0.676)} | jumlah = 1.0


### §7.4 Pembanding independen: scikit-learn

Implementasi manual tidak harus identik bit-per-bit dengan scikit-learn (detail
tie-breaking split berbeda), tapi harus **setara**. Kalau selisihnya besar, ada bug di
implementasi manual — bukan "beda library".

Sel ini hanya verifikasi; model yang diekspor tetap yang manual.

In [44]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score

if model_type == "knn":
    ref = KNeighborsClassifier(
        n_neighbors=best_params["k"], weights=best_params["weights"],
        metric=best_params["metric"], algorithm="brute",
    ).fit(X_tr, y_tr)
else:
    ref = RandomForestClassifier(
        n_estimators=best_params["n_estimators"], max_depth=best_params["max_depth"],
        max_features=best_params.get("max_features", "sqrt"),
        random_state=SEED, n_jobs=-1,
    ).fit(X_tr, y_tr)

ref_f1 = f1_score(y_te, ref.predict(X_te), average="macro", zero_division=0)
print(f"macro-F1 implementasi manual : {final_metrics['macro_f1']:.4f}")
print(f"macro-F1 scikit-learn        : {ref_f1:.4f}")
print(f"selisih                      : {abs(final_metrics['macro_f1'] - ref_f1):.4f}")

macro-F1 implementasi manual : 0.5327
macro-F1 scikit-learn        : 0.5251
selisih                      : 0.0077


## §8. Laporan dan kesimpulan

Bagian ini merangkum hasil dalam bentuk yang bisa ditempel ke laporan: model aktif,
perbandingan, metrik, dan batasannya. Tidak ada perhitungan baru di sini — semuanya
dibaca dari `model_meta.json` yang baru ditulis, sehingga angka laporan dan angka yang
dipakai backend dijamin sama.

### §8.1 Model aktif dan perbandingan

Semua angka di bawah dibaca langsung dari `model_meta.json`, jadi tidak mungkin berbeda
dengan yang dilihat backend.

In [45]:
print("=" * 78)
print("RINGKASAN AKHIR")
print("=" * 78)
print(f"Model aktif    : {meta['model_name']} ({meta['model_type']})")
print(f"Hyperparameter : {meta['best_params']}")
print(f"Comparator     : {meta['comparator_info']['model_name']} -> {meta['comparator_info']['best_params']}")
print(f"Data           : {meta['n_train']:,} latih / {meta['n_test']:,} uji (holdout {meta['holdout_ratio']:.0%})")
print(f"d_enc          : {meta['d_enc']} kolom setelah preprocessing")

RINGKASAN AKHIR
Model aktif    : Random Forest (random_forest)
Hyperparameter : {'n_estimators': 300, 'max_depth': 20, 'max_features': None}
Comparator     : KNN -> {'k': 5, 'weights': 'distance', 'metric': 'manhattan'}
Data           : 8,118 latih / 2,027 uji (holdout 20%)
d_enc          : 60 kolom setelah preprocessing


### §8.2 Metrik dan alasan pemilihan

Metrik dilaporkan lengkap (bukan hanya macro-F1) supaya pembaca bisa menilai sendiri:
model ini bagus di kelas mayoritas, lemah di kelas yang paling penting secara operasional.

In [46]:
m = meta["evaluation_metrics"]
print(f"macro-F1       : {m['macro_f1']:.4f}")
print(f"balanced acc   : {m['balanced_accuracy']:.4f}")
print(f"akurasi        : {m['accuracy']:.4f}")
print("recall per kelas:")
for label in CLASS_LABELS:
    print(f"  {label:8s} {m['per_class'][label]['recall']:.4f}  (support {m['per_class'][label]['support']})")

print("\nKenapa model ini yang dipilih:")
print(f"  rata-rata macro-F1 di 3 rasio holdout {mean_f1[winner]:.4f} vs "
      f"{mean_f1[loser]:.4f} ({loser});")
print("  parameter diekspor dari rasio primer, bukan rasio dengan skor terbaik;")
print("  hasil latih ulang identik dengan hasil eksperimen (deterministik, seed tetap).")

macro-F1       : 0.5327
balanced acc   : 0.5195
akurasi        : 0.7218
recall per kelas:
  small    0.5898  (support 551)
  medium   0.8568  (support 1306)
  large    0.1118  (support 170)

Kenapa model ini yang dipilih:
  rata-rata macro-F1 di 3 rasio holdout 0.5361 vs 0.5085 (KNN);
  parameter diekspor dari rasio primer, bukan rasio dengan skor terbaik;
  hasil latih ulang identik dengan hasil eksperimen (deterministik, seed tetap).


### §8.3 Batasan yang harus ditulis di laporan

Bagian ini sengaja tidak mempercantik hasil. Model dengan macro-F1 ±0,50 adalah model yang
jujur soal keterbatasan fiturnya, dan itu harus tertulis.

In [47]:
print("Batasan:")
print("  - Fitur hanya musim + posisi bola + 4 kategori; tidak ada curah hujan, kemiringan,")
print("    atau litologi, sehingga plafon akurasi memang terbatas.")
print(f"  - Kelas 'large' tetap sulit: recall {m['per_class']['large']['recall']:.2f}. Model")
print("    cenderung aman memprediksi kelas mayoritas; untuk peringatan dini ini perlu")
print("    penanganan khusus (mis. bobot kelas atau threshold per kelas).")
print("  - Evaluasi memakai data historis GLC, bukan data operasional lapangan.")

Batasan:
  - Fitur hanya musim + posisi bola + 4 kategori; tidak ada curah hujan, kemiringan,
    atau litologi, sehingga plafon akurasi memang terbatas.
  - Kelas 'large' tetap sulit: recall 0.11. Model
    cenderung aman memprediksi kelas mayoritas; untuk peringatan dini ini perlu
    penanganan khusus (mis. bobot kelas atau threshold per kelas).
  - Evaluasi memakai data historis GLC, bukan data operasional lapangan.


## §A. Lampiran — profil biaya komputasi (opsional)

Bagian ini **tidak** menghasilkan artefak dan boleh dilewati. Isinya menjawab pertanyaan
yang muncul saat grid search terasa lambat: berapa biaya satu pohon, apakah `n_jobs`
menolong, dan seberapa dalam pohon yang tumbuh tanpa batas.

Angka di sini juga yang jadi dasar pemilihan `N_JOBS` dan optimasi prefix pohon di §4.2.

### §A.1 Siapkan potongan data satu fold

Dipakai rasio tengah (0.15) supaya ukurannya representatif, lalu diambil 80% pertama
sebagai porsi latih satu fold — meniru kondisi sebenarnya saat CV berjalan.

In [48]:
y_prof = encode_target(df_model[TARGET].to_numpy())
tr_prof, _, ytr_prof, _ = stratified_holdout(df_model, y_prof, 0.15, SEED)  # rasio tengah
X_prof = Preprocessor().fit_transform(tr_prof)
n_fold_train = int(len(X_prof) * 0.8)                 # porsi latih satu fold
X_fold, y_fold = X_prof[:n_fold_train], ytr_prof[:n_fold_train]
print(f"Data latih per fold: {X_fold.shape[0]:,} baris x {X_fold.shape[1]} kolom")

Data latih per fold: 6,900 baris x 60 kolom


### §A.2 Biaya per pohon menurut `max_depth`

Pohon tanpa batas kedalaman (`None`) jauh lebih mahal karena terus membelah sampai murni.
Ini alasan `max_depth` ikut masuk grid, bukan hanya `n_estimators`.

In [49]:
print("Biaya per pohon (4 pohon per kedalaman, n_jobs=1):")
per_tree = {}
for depth in RF_GRID["max_depth"]:
    t0 = time.time()
    RandomForest(n_estimators=4, max_depth=depth, random_state=SEED, n_jobs=1).fit(X_fold, y_fold)
    per_tree[depth] = (time.time() - t0) / 4
    print(f"  max_depth={str(depth):>4}: {per_tree[depth]:6.2f} detik/pohon")

Biaya per pohon (4 pohon per kedalaman, n_jobs=1):
  max_depth=None:   0.22 detik/pohon
  max_depth=  10:   0.09 detik/pohon
  max_depth=  20:   0.22 detik/pohon
  max_depth=  30:   0.23 detik/pohon


### §A.3 Apakah `n_jobs` menolong?

RF manual memecah pohon ke beberapa proses. Pengukurannya di sini menentukan apakah
paralelisme layak dipakai di mesin ini. Kalau lingkungan eksekusi tidak mendukung proses
anak (mis. skrip tanpa guard `__main__`), bagian ini dilewati dengan pesan jelas dan
estimasi tetap memakai `n_jobs=1`.

In [50]:
nj_time = {}
t0 = time.time()
RandomForest(n_estimators=12, max_depth=None, random_state=SEED, n_jobs=1).fit(X_fold, y_fold)
nj_time[1] = (time.time() - t0) / 12
print(f"  n_jobs=1: {nj_time[1]:6.2f} detik/pohon efektif")

try:
    for nj in (2, 3, 4):
        t0 = time.time()
        RandomForest(n_estimators=12, max_depth=None, random_state=SEED, n_jobs=nj).fit(X_fold, y_fold)
        nj_time[nj] = (time.time() - t0) / 12
        print(f"  n_jobs={nj}: {nj_time[nj]:6.2f} detik/pohon efektif")
except Exception as exc:
    print(f"  n_jobs>1 dilewati: {type(exc).__name__}: {exc}")
    print("  (proses anak gagal start di lingkungan ini; jalankan di Jupyter/terminal)")

best_nj = min(nj_time, key=nj_time.get)
print(f"\nn_jobs tercepat: {best_nj} (percepatan {nj_time[1] / nj_time[best_nj]:.2f}x vs 1)")

  n_jobs=1:   0.23 detik/pohon efektif
  n_jobs=2:   0.20 detik/pohon efektif
  n_jobs=3:   0.15 detik/pohon efektif
  n_jobs=4:   0.15 detik/pohon efektif

n_jobs tercepat: 3 (percepatan 1.55x vs 1)


### §A.4 Kedalaman pohon dan estimasi total

`depth=None` berarti pohon tumbuh sampai tidak bisa membelah lagi. Kedalaman aktualnya
menunjukkan seberapa besar overfitting yang mungkin terjadi, dan estimasi total menjelaskan
kenapa grid penuh bisa memakan waktu berjam-jam.

In [51]:
def tree_depths(tree):
    # Anak selalu punya indeks lebih besar dari induk, jadi satu kali lewat cukup.
    depth = np.zeros(len(tree.feature_idx_), dtype=int)
    for i in range(len(depth)):
        if tree.left_child_[i] != -1:
            depth[tree.left_child_[i]] = depth[i] + 1
            depth[tree.right_child_[i]] = depth[i] + 1
    return depth.max()


probe = RandomForest(n_estimators=20, max_depth=None, random_state=SEED, n_jobs=1).fit(X_fold, y_fold)
d = np.array([tree_depths(t) for t in probe.trees_])
print(f"Kedalaman pohon (depth=None): min {d.min()}, median {int(np.median(d))}, maks {d.max()}")
print(f"Persentil 90/99             : {int(np.percentile(d, 90))} / {int(np.percentile(d, 99))}")

# Estimasi: 300 pohon per (max_features, max_depth) per fold, x fold, x rasio.
# Prefix cache menyimpan satu hutan per kombinasi parameter selain n_estimators,
# jadi tiap nilai max_features menambah satu hutan penuh per fold.
n_mf = len(RF_GRID.get("max_features", [None]))
per_fold_seconds = sum(
    per_tree[dep] * max(RF_GRID["n_estimators"]) * n_mf
    for dep in RF_GRID["max_depth"]
)
print()
for nj in sorted(nj_time):
    speedup = nj_time[1] / nj_time[nj]
    total = per_fold_seconds * N_SPLITS * len(TEST_SIZES) / speedup
    print(f"Estimasi grid RF penuh, n_jobs={nj}: {total / 3600:5.1f} jam")
print(f"Run aktual memakai N_JOBS={N_JOBS}; angka n_jobs lain hanya ilustrasi.")
if None in RF_GRID.get("max_features", []):
    print("Catatan: per_tree diukur pada max_features='sqrt'. max_features=None "
          "mengevaluasi 60 kolom per simpul dan terukur ~4x lebih mahal per pohon, "
          "jadi angka di atas adalah BATAS BAWAH; kalikan ~2,5x untuk perkiraan "
          "realistis (cocok dengan durasi run aktual di §4.2).")


Kedalaman pohon (depth=None): min 25, median 28, maks 35
Persentil 90/99             : 31 / 34

Estimasi grid RF penuh, n_jobs=1:   1.9 jam
Estimasi grid RF penuh, n_jobs=2:   1.6 jam
Estimasi grid RF penuh, n_jobs=3:   1.2 jam
Estimasi grid RF penuh, n_jobs=4:   1.2 jam
Run aktual memakai N_JOBS=4; angka n_jobs lain hanya ilustrasi.
Catatan: per_tree diukur pada max_features='sqrt'. max_features=None mengevaluasi 60 kolom per simpul dan terukur ~4x lebih mahal per pohon, jadi angka di atas adalah BATAS BAWAH; kalikan ~2,5x untuk perkiraan realistis (cocok dengan durasi run aktual di §4.2).
